# MNIST Digit Classifier — Computer Vision Project

### 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical

### 2. Load training data

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train shape", X_train.shape)
print("y_train shape", y_train.shape)

### 3. Preprocess training data

In [ ]:
X_train = X_train.reshape(60000, 784).astype('float32') / 255

no_classes = 10
Y_train = to_categorical(y_train, no_classes)

### 4. Build model

In [ ]:
model = Sequential()

model.add(Dense(512, input_shape=(784,)))
model.add(Activation('relu'))
model.add(Dropout(0.2))

model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.2))

model.add(Dense(10))
model.add(Activation('softmax'))

model.summary()

### 5. Compile model

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

### 6. Train model

In [ ]:
history = model.fit(X_train, Y_train,
                    batch_size=128, epochs=10,
                    verbose=1)

### 7. Upload your input data
Run this single cell. In real Google Colab this shows a native **Choose Files** button — click it, pick `sample_mnist_test.csv`, and wait for the upload progress bar to finish. The same cell then loads the file automatically, no second cell needed.

In [ ]:
data = None
filename = None

try:
    # Native Colab upload (recommended — only works on colab.research.google.com)
    from google.colab import files
    print("Click 'Choose Files' below and select sample_mnist_test.csv")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("No file was selected.")
    filename = list(uploaded.keys())[0]
    data = pd.read_csv(io.BytesIO(uploaded[filename]))

except ImportError:
    # Fallback for non-Colab notebooks (VS Code, JupyterLab, classic Jupyter)
    import ipywidgets as widgets
    from IPython.display import display

    upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
    display(upload_widget)
    print("Select a file above, then RE-RUN this same cell once to load it.")

    if len(upload_widget.value) > 0:
        item = upload_widget.value[0] if not isinstance(upload_widget.value, dict) \
            else list(upload_widget.value.values())[0]
        filename = item.get('name') or item.get('metadata', {}).get('name')
        content = bytes(item['content'])
        data = pd.read_csv(io.BytesIO(content))

if data is not None:
    print("Loaded:", filename, "-", data.shape[0], "rows,", data.shape[1], "columns")

### 8. Preprocess the uploaded data

In [ ]:
assert data is not None, "No data loaded yet — go back to Step 7, upload a file, then re-run Step 7 and this cell."

if 'label' in data.columns:
    true_labels = data['label'].values
    pixels = data.drop('label', axis=1).values
else:
    true_labels = None
    pixels = data.values

assert pixels.shape[1] == 784, f"Expected 784 pixel columns, got {pixels.shape[1]}. Check your CSV format."

pixels = pixels.astype('float32') / 255
print("Preprocessed", pixels.shape[0], "images.")

### 9. Predict on the uploaded data

In [ ]:
probs = model.predict(pixels)
predicted_classes = np.argmax(probs, axis=1)

print("Predictions:", predicted_classes[:20])
if true_labels is not None:
    accuracy = (predicted_classes == true_labels).mean()
    print("Accuracy on uploaded data:", accuracy)

### 10. Visualize predictions

In [ ]:
plt.figure()
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(pixels[i].reshape(28, 28), cmap='gray', interpolation='none')
    title = "Pred {}".format(predicted_classes[i])
    if true_labels is not None:
        title += ", True {}".format(true_labels[i])
    plt.title(title)
plt.tight_layout()